# Gerando um CSV na landing-zone a partir do banco

### Carregando as variáveis de ambiente

In [1]:
import os, io
import pandas as pd
import pyodbc
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv(override=True)

DB_HOST = os.getenv("HOST")
DB_PORT = os.getenv("PORT")
DB_NAME = os.getenv("DATABASE")
DB_USER = os.getenv("USER")
DB_PASS = os.getenv("PASSWORD")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")
MINIO_LANDING_BUCKET = os.getenv("MINIO_LANDING_BUCKET")
MINIO_BRONZE_BUCKET = os.getenv("MINIO_BRONZE_BUCKET")

print(f"PostgreSQL: {DB_HOST}:{DB_PORT}/{DB_NAME}")
print(f"MinIO: {MINIO_ENDPOINT} | Bucket: {MINIO_LANDING_BUCKET}")

PostgreSQL: 127.0.0.1:5432/delta_games
MinIO: http://localhost:9000 | Bucket: landing-zone


### Conectando ao Postgres

In [2]:
conn = pyodbc.connect(
    "DRIVER={PostgreSQL UNICODE};"
    f"SERVER={DB_HOST},{DB_PORT};"
    f"DATABASE={DB_NAME};"
    f"UID={DB_USER};"
    f"PWD={DB_PASS};"
    f"TrustServerCertificate=yes;"
)

cursor = conn.cursor()

print(f"Conectado ao postgres: {DB_NAME}")

Conectado ao postgres: delta_games


### Listando as tabelas

In [3]:
sql = "SELECT table_name FROM information_schema.tables WHERE table_schema = 'public' AND table_type = 'BASE TABLE' ORDER BY table_name;"

cursor.execute(sql)
result = cursor.fetchall()
tables = [line.table_name for line in result]

print(f"Tabelas encontradas: {len(tables)}")
print("--" * 20)

for i, t in enumerate(tables, start=1):
    cursor.execute(f"SELECT COUNT(*) FROM public.{t};")

    print(f"{i:2d}. {t:<20} ({cursor.fetchone()[0]} registros)")

Tabelas encontradas: 5
----------------------------------------
 1. games                (21910 registros)
 2. how_long_to_beat     (13196 registros)
 3. regional_sales       (16608 registros)
 4. steam_metrics        (3178 registros)
 5. walkthrough_links    (688 registros)


### Criando o Bucket no MinIO

In [4]:
s3 = boto3.client(
    "s3",
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version="s3v4"),
    region_name="us-east-1"
)

try:
    s3.head_bucket(Bucket=MINIO_LANDING_BUCKET)
    print(f"Bucket: {MINIO_LANDING_BUCKET} já existe.")
except:
    s3.create_bucket(Bucket=MINIO_LANDING_BUCKET)
    print(f"Bucket: {MINIO_LANDING_BUCKET} criado.")


print("Buckets: ", [bucket["Name"] for bucket in s3.list_buckets()["Buckets"]])

Bucket: landing-zone criado.
Buckets:  ['landing-zone']


### Extrair tabelas para CSV no MinIO

In [5]:
results = []

for t in tables:
    cursor.execute(f"SELECT * FROM {t};")
    columns = [desc[0] for desc in cursor.description]
    rows = cursor.fetchall()
    df = pd.DataFrame.from_records(rows, columns=columns)

    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False)
    csv_bytes = csv_buffer.getvalue().encode("utf-8")
    s3_key = f"{t}.csv"

    s3.put_object(Bucket=MINIO_LANDING_BUCKET, Key=s3_key, Body=csv_bytes, ContentType="text/csv")

    size_kb = len(csv_bytes) / 1024
    results.append({"table": t, "registros": len(df), "colunas": len(df.columns), "tamanho_kb": round(size_kb, 1)})
    print(f"{t}.csv -> s3://{MINIO_LANDING_BUCKET}/{s3_key} ({len(df)} registros, {size_kb:.1f} KB)")

print(f"\nExtração concluída: {len(tables)} tabelas exportadas.")

games.csv -> s3://landing-zone/games.csv (21910 registros, 2355.9 KB)
how_long_to_beat.csv -> s3://landing-zone/how_long_to_beat.csv (13196 registros, 289.3 KB)
regional_sales.csv -> s3://landing-zone/regional_sales.csv (16608 registros, 495.6 KB)
steam_metrics.csv -> s3://landing-zone/steam_metrics.csv (3178 registros, 103.7 KB)
walkthrough_links.csv -> s3://landing-zone/walkthrough_links.csv (688 registros, 46.4 KB)

Extração concluída: 5 tabelas exportadas.


### Validando o resultado

In [6]:
response = s3.list_objects_v2(Bucket=MINIO_LANDING_BUCKET)
print(f"Arquivos no Bucket [{MINIO_LANDING_BUCKET}]:\n")

res = response.get("Contents", [])

for obj in res:
    print(f"{obj['Key']:<25} {obj['Size']/1024:>8.1f} KB")

df_resumo = pd.DataFrame(results)
print(f"\nTotal: {df_resumo['registros'].sum():,} registros | {df_resumo['tamanho_kb'].sum():,.1f} KB")

Arquivos no Bucket [landing-zone]:

games.csv                   2355.9 KB
how_long_to_beat.csv         289.3 KB
regional_sales.csv           495.6 KB
steam_metrics.csv            103.7 KB
walkthrough_links.csv         46.4 KB

Total: 55,580 registros | 3,290.9 KB


### Encerrando a conexão

In [7]:
cursor.close()
conn.close()

print("Conexão com o banco encerrada")

Conexão com o banco encerrada
